# Practical 10: Feature Scaling (StandardScaler vs MinMaxScaler) & Train-Test Split

**UNIT 3: Preparing Big Data for Machine Learning**

**Problem Statement:** Write a Python program to perform feature scaling and dataset partitioning. Apply StandardScaler (Z-score normalization) and MinMaxScaler to numerical features, compare their transformed distributions, and split the dataset into training (80%) and testing (20%) sets using train_test_split(..., test_size=0.2, random_state=42).

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 200)
rs = np.random.default_rng(11)
n = 150
age = rs.integers(21, 59, n).astype(float)
salary = np.round(25500 + age * 890 + rs.normal(0, 6000, n), -2)
data = pd.DataFrame({
    "Country": rs.choice(["India", "Japan", "Brazil"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + rs.normal(0, 8000, n) > 53500, "Yes", "No"),
})
data.loc[rs.choice(n, 8, replace=False), "Age"] = np.nan
data.loc[rs.choice(n, 8, replace=False), "Salary"] = np.nan
print(data.head(9))

num_cols = ["Age", "Salary"]
data[num_cols] = data[num_cols].fillna(data[num_cols].mean())
print(data[num_cols].isna().sum())

z_scaled = pd.DataFrame(StandardScaler().fit_transform(data[num_cols]), columns=num_cols)
minmax_scaled = pd.DataFrame(MinMaxScaler().fit_transform(data[num_cols]), columns=num_cols)

stats = ["mean", "std", "min", "max"]
print("=== Original ===")
print(data[num_cols].describe().loc[stats].round(3))
print("\n=== StandardScaler (mean ≈ 0, std ≈ 1) ===")
print(z_scaled.describe().loc[stats].round(3))
print("\n=== MinMaxScaler (range [0, 1]) ===")
print(minmax_scaled.describe().loc[stats].round(3))

X = pd.get_dummies(data.drop(columns=["Purchased"]), columns=["Country"], dtype=int)
y = data["Purchased"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
print(f"Total rows: {len(X)} | Train: {X_train.shape} | Test: {X_test.shape}")

sc = StandardScaler()
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[num_cols] = sc.fit_transform(X_train[num_cols])
X_test_s[num_cols] = sc.transform(X_test[num_cols])

print("Scaled TRAIN stats (mean ≈ 0, std ≈ 1):")
print(X_train_s[num_cols].describe().loc[["mean", "std"]].round(3))
print("\nScaled TEST stats (close to, but not exactly, 0/1 – as expected):")
print(X_test_s[num_cols].describe().loc[["mean", "std"]].round(3))

,Country,Age,Salary,Purchased
0,India,26.0,56200.0,Yes
1,Brazil,25.0,58500.0,Yes
2,Brazil,51.0,NaN,No
3,Japan,39.0,65500.0,Yes
4,Japan,43.0,66600.0,Yes
5,Japan,43.0,63200.0,Yes
6,Japan,48.0,62200.0,No
7,Brazil,22.0,52600.0,Yes
8,Japan,39.0,52600.0,Yes


Age       0
Salary    0
dtype: int64
=== Original ===

,Age,Salary
mean,39.507,60587.324
std,11.558,11625.036
min,21.000,34400.000
max,58.000,84400.000



=== StandardScaler (mean ≈ 0, std ≈ 1) ===

,Age,Salary
mean,-0.000,0.000
std,1.003,1.003
min,-1.607,-2.260
max,1.605,2.055


...

,Age,Salary
mean,0.500,0.524
std,0.312,0.233
min,0.000,0.000
max,1.000,1.000


Total rows: 150 | Train: (105, 5) | Test: (45, 5)
Scaled TRAIN stats (mean ≈ 0, std ≈ 1):

,Age,Salary
mean,-0.000,-0.000
std,1.005,1.005



Scaled TEST stats (close to, but not exactly, 0/1 – as expected):

,Age,Salary
mean,0.55,0.306
std,0.95,0.850
